# Workforce Bridge Program — Intern Pipeline Analysis
## Predicting Attrition and Identifying High-Quality Employer Partners

**Dataset:** Workforce Bridge Program (WBP) — anonymized per FERPA  
**Scope:** 1,193 intern placements across 5 campuses, FY2020–FY2024  
**Tools:** Python · pandas · matplotlib · seaborn · scikit-learn

---

### Project Overview

This notebook presents an end-to-end people analytics investigation of
four and a half years of internship placement data from a multi-campus
workforce development program at a public community college district.

The analysis is organized into three connected parts:

- **Part 1 — Pipeline Funnel Analysis:** Where do students exit the program, and when?
- **Part 2 — Employer Quality Analysis:** Which employer partners drive the best outcomes?
- **Part 3 — Policy Recommendations:** What operational changes does the data support?

> **Data privacy:** All student identifiers have been anonymized per FERPA.
> Student names, institutional IDs, and contact information are replaced
> with coded equivalents. Employer names are retained as organizational
> entities. Campus names are replaced with Campus A through Campus E.
> The institution is not identified.


---
## 0. Environment Setup

In [ ]:
# Uncomment if running in Colab and openpyxl is not installed
# !pip install openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       '#E0E0E0',
    'grid.linewidth':   0.7,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

PALETTE = {
    'Planned Completion': '#2ECC71',
    'Re-engagement':      '#3498DB',
    'Student Exit':       '#E74C3C',
    'Employer Exit':      '#E67E22',
    'Administrative Exit':'#95A5A6',
}

print("Environment ready.")


### 0.1 Load the Anonymized Dataset

Upload `WBP_Anonymized.xlsx` to Colab using the file panel on the left,
then run this cell.


In [ ]:
FILE = 'WBP_Anonymized.xlsx'

xl = pd.ExcelFile(FILE)
print("Sheets found:", xl.sheet_names)

placements  = pd.read_excel(FILE, sheet_name='Total Placements')
daily_hired = pd.read_excel(FILE, sheet_name='Daily Hired Report')
ended_curr  = pd.read_excel(FILE, sheet_name='Ended Assignments')
ended_arch  = pd.read_excel(FILE, sheet_name='ARCHIVED-Ended Assignments')
on_assign   = pd.read_excel(FILE, sheet_name='Data Currently on Assignment')
ncns        = pd.read_excel(FILE, sheet_name='PLACED-No Call and No Shows')

print(f"\nRecord counts:")
print(f"  Total Placements:           {len(placements):>5}")
print(f"  Daily Hired Report:         {len(daily_hired):>5}")
print(f"  Ended Assignments:          {len(ended_curr):>5}")
print(f"  ARCHIVED-Ended Assignments: {len(ended_arch):>5}")
print(f"  Currently on Assignment:    {len(on_assign):>5}")
print(f"  No Call No Shows:           {len(ncns):>5}")


In [ ]:
# Preview column names — used to build the rename mapping below
print("=== Total Placements columns ===")
print(list(placements.columns))
print("\n=== Ended Assignments columns ===")
print(list(ended_curr.columns))


---
## Part 1 — Data Cleaning & Outcome Classification

### 1.1 Standardize Column Names

Edit the mapping dictionaries below to match your actual column names
as printed by the cell above.


In [ ]:
# ── Column rename mapping ────────────────────────────────────────────────────
# Left  = what the Excel file actually has
# Right = what we use throughout this notebook

PLACEMENT_COLS = {
    'Student ID':     'student_id',
    'Student Name':   'student_name',
    'ACES ID':        'student_code',
    'Banner ID':      'system_id',
    'College':        'campus',
    'Employer':       'employer',
    'Start Date':     'start_date',
    'End Date':       'end_date',
    'Status':         'status',
    'Reason':         'reason_code',
    'Semester':       'semester',
    'Academic Year':  'academic_year',
    'Major':          'major',
    'Hours Per Week': 'hours_per_week',
}

ENDED_COLS = {
    'Student ID':     'student_id',
    'Student Name':   'student_name',
    'ACES ID':        'student_code',
    'College':        'campus',
    'Employer':       'employer',
    'Start Date':     'start_date',
    'End Date':       'end_date',
    'Reason':         'reason_code',
    'Semester':       'semester',
    'Academic Year':  'academic_year',
}

def safe_rename(df, mapping):
    existing = {k: v for k, v in mapping.items() if k in df.columns}
    return df.rename(columns=existing)

placements = safe_rename(placements, PLACEMENT_COLS)
ended_curr = safe_rename(ended_curr, ENDED_COLS)
ended_arch = safe_rename(ended_arch, ENDED_COLS)

print("Renaming complete.")
print("Placements columns:", list(placements.columns))


### 1.2 Outcome Classification Framework

The five categories below reflect how program outcomes were understood
operationally. The raw reason codes carry institutional meaning that is
not visible from the codes alone.

| Category | Meaning |
|---|---|
| **Planned Completion** | Student finished as intended — term ended, graduated, or was hired |
| **Re-engagement** | Student left to return for another placement — a positive signal |
| **Student Exit** | Student-driven early departure |
| **Employer Exit** | Employer-driven early departure |
| **Administrative Exit** | Program-level action unrelated to student or employer performance |

> **Note on structured placements:** Some employer partners run fixed-length
> programs (for example, a structured 9-week placement). Their
> `Internship Term Ended` records reflect planned completion, not dropout.
> These records are flagged with `is_structured_program = True` and
> excluded from attrition timing calculations to avoid distorting the
> distribution.


In [ ]:
# ── Outcome classifier ───────────────────────────────────────────────────────

# Employers with structured fixed-length programs
# Add any others you know belong in this category
STRUCTURED_EMPLOYER_KEYWORDS = [
    'Accenture',         # 9-week structured placement
    'MIMS Institute',    # structured short-term
]

def classify_outcome(row):
    reason  = str(row.get('reason_code', '')).strip().lower()
    planned = [
        'internship term ended', 'term ended', 'graduate',
        'hired by employer', 'hired permanently',
        'hired by non-wbp employer permanently',
    ]
    reengage = [
        'seeking new wbp assignment', 'seeking new assignment',
        'seeking new aotj assignment',   # legacy code in dataset
    ]
    student_exit = [
        'voluntary withdrawal', 'student ended assignment',
        'no call no show', 'did not complete employer pre-screen',
        'student withdrew', 'abandoned',
    ]
    employer_exit = [
        'employer ended assignment', 'employer terminated',
    ]
    admin_exit = [
        'ineligible-not enrolled', 'wbp pause', 'aotj pause',
        'administrative hold', 'program ended',
    ]
    for c in planned:
        if c in reason: return 'Planned Completion'
    for c in reengage:
        if c in reason: return 'Re-engagement'
    for c in student_exit:
        if c in reason: return 'Student Exit'
    for c in employer_exit:
        if c in reason: return 'Employer Exit'
    for c in admin_exit:
        if c in reason: return 'Administrative Exit'
    return 'Unclassified'

def flag_structured(row):
    emp = str(row.get('employer', '')).strip()
    return any(kw.lower() in emp.lower() for kw in STRUCTURED_EMPLOYER_KEYWORDS)

ended_curr['outcome'] = ended_curr.apply(classify_outcome, axis=1)
ended_arch['outcome'] = ended_arch.apply(classify_outcome, axis=1)
ended_curr['is_structured_program'] = ended_curr.apply(flag_structured, axis=1)
ended_arch['is_structured_program'] = ended_arch.apply(flag_structured, axis=1)

ended_all = pd.concat([ended_curr, ended_arch], ignore_index=True)

unclassified = (ended_all['outcome'] == 'Unclassified').sum()
total_ended  = len(ended_all)
print(f"Total ended assignments:  {total_ended}")
print(f"Classified:               {total_ended - unclassified}  ({(1 - unclassified/total_ended)*100:.1f}%)")
print(f"Unclassified:             {unclassified}")

if unclassified > 0:
    print("\nUnclassified reason codes (add these to the classifier above):")
    print(ended_all[ended_all['outcome'] == 'Unclassified']['reason_code'].value_counts())


In [ ]:
# ── Date parsing and tenure calculation ──────────────────────────────────────
for df in [ended_all, placements]:
    for col in ['start_date', 'end_date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

ended_all['tenure_days'] = (
    ended_all['end_date'] - ended_all['start_date']
).dt.days

neg  = (ended_all['tenure_days'] < 0).sum()
null = ended_all['tenure_days'].isna().sum()
print(f"Tenure calculated: {total_ended - null} records")
print(f"Negative tenure (data quality flag): {neg}")
print(f"Missing tenure:                      {null}")
print(f"\nMedian tenure: {ended_all['tenure_days'].median():.0f} days")
print(f"Mean tenure:   {ended_all['tenure_days'].mean():.0f} days")


In [ ]:
# ── Final distribution check ─────────────────────────────────────────────────
print("=== Outcome Distribution ===")
outcome_counts = ended_all['outcome'].value_counts()
for outcome, count in outcome_counts.items():
    pct = count / len(ended_all) * 100
    print(f"  {outcome:<25} {count:>4}  ({pct:.1f}%)")

print(f"\n=== Structured Program Records ===")
print(ended_all['is_structured_program'].value_counts().to_string())

if 'campus' in ended_all.columns:
    print(f"\n=== Campus Distribution ===")
    print(ended_all['campus'].value_counts().to_string())


---
## Part 2 — Student Pipeline Funnel Analysis

### Research Question
*At what stage do students exit the Workforce Bridge Program pipeline,
and which campuses and cohorts have the highest attrition rates?*


### 2.1 Overall Outcome Distribution

In [ ]:
outcome_order = [
    'Planned Completion', 'Re-engagement',
    'Student Exit', 'Employer Exit', 'Administrative Exit', 'Unclassified'
]
counts = ended_all['outcome'].value_counts().reindex(outcome_order, fill_value=0)
colors = [PALETTE.get(o, '#BDC3C7') for o in outcome_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Workforce Bridge Program — Intern Outcome Distribution\n'
    '4.5 Years · 5 Campuses · 1,193 Placements',
    fontsize=15, fontweight='bold', y=1.01
)

# Horizontal bar chart
ax = axes[0]
bars = ax.barh(counts.index, counts.values, color=colors,
               edgecolor='white', linewidth=1.2, height=0.65)
ax.set_xlabel('Number of Placements')
ax.set_title('Count by Outcome')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=10)

# Donut chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    counts.values, labels=counts.index, colors=colors,
    autopct=lambda p: f'{p:.1f}%' if p > 2 else '',
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
    pctdistance=0.75,
)
centre_circle = plt.Circle((0, 0), 0.55, fc='white')
ax2.add_artist(centre_circle)
ax2.set_title('Proportion by Outcome')
ax2.text(0, 0, f'n={counts.sum()}', ha='center', va='center',
         fontsize=13, fontweight='bold', color='#2C3E50')

plt.tight_layout()
plt.savefig('fig1_outcome_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")


### 2.2 Completion Rate by Campus

In [ ]:
if 'campus' in ended_all.columns:
    campus_outcomes = (
        ended_all.groupby(['campus', 'outcome'])
        .size().unstack(fill_value=0)
        .reindex(columns=outcome_order, fill_value=0)
    )
    campus_totals = campus_outcomes.sum(axis=1)
    campus_pct    = campus_outcomes.div(campus_totals, axis=0) * 100
    campus_pct_sorted = campus_pct.sort_values('Planned Completion', ascending=False)

    fig, ax = plt.subplots(figsize=(12, 6))
    campus_pct_sorted[outcome_order].plot(
        kind='bar', stacked=True, ax=ax,
        color=[PALETTE.get(o, '#BDC3C7') for o in outcome_order],
        edgecolor='white', linewidth=0.8, width=0.7
    )
    ax.set_title('Intern Outcome by Campus\n(sorted by completion rate)', pad=12)
    ax.set_xlabel('')
    ax.set_ylabel('Percentage of Placements')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1), framealpha=0.9)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

    for i, campus in enumerate(campus_pct_sorted.index):
        ax.text(i, 102, f'n={campus_totals[campus]}',
                ha='center', va='bottom', fontsize=9, color='#555')

    plt.tight_layout()
    plt.savefig('fig2_outcome_by_campus.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nCompletion rates by campus:")
    for campus in campus_pct_sorted.index:
        rate = campus_pct_sorted.loc[campus, 'Planned Completion']
        n    = campus_totals[campus]
        print(f"  {campus:<30} {rate:.1f}%  (n={n})")
else:
    print("No 'campus' column — check column rename mapping.")


### 2.3 Attrition Timing — When Do Students Exit?

In [ ]:
exits = ended_all[
    (ended_all['outcome'] == 'Student Exit') &
    (~ended_all['is_structured_program']) &
    (ended_all['tenure_days'].notna()) &
    (ended_all['tenure_days'] >= 0)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'When Do Student Exits Occur?\n(Structured employer programs excluded)',
    fontsize=14, fontweight='bold'
)

ax = axes[0]
ax.hist(exits['tenure_days'], bins=30, color='#E74C3C', alpha=0.8, edgecolor='white')
med = exits['tenure_days'].median()
ax.axvline(med,  color='#2C3E50', linestyle='--', linewidth=2, label=f'Median: {med:.0f} days')
ax.axvline(30,   color='#E67E22', linestyle=':',  linewidth=1.5, label='30 days')
ax.axvline(90,   color='#F39C12', linestyle=':',  linewidth=1.5, label='90 days')
ax.set_xlabel('Days on Assignment Before Exit')
ax.set_ylabel('Number of Students')
ax.set_title('Exit Timing Distribution')
ax.legend(fontsize=9)

ax2 = axes[1]
sorted_days = np.sort(exits['tenure_days'])
cumulative  = np.arange(1, len(sorted_days) + 1) / len(sorted_days) * 100
ax2.plot(sorted_days, cumulative, color='#E74C3C', linewidth=2.5)
ax2.axhline(50, color='#7F8C8D', linestyle='--', linewidth=1, alpha=0.7)
ax2.axvline(med, color='#2C3E50', linestyle='--', linewidth=1.5,
            label=f'50% exited by day {med:.0f}')
ax2.set_xlabel('Days on Assignment')
ax2.set_ylabel('Cumulative % of Exits')
ax2.set_title('Cumulative Exit Rate')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.legend(fontsize=9)

early_30 = (exits['tenure_days'] <= 30).mean() * 100
early_60 = (exits['tenure_days'] <= 60).mean() * 100
early_90 = (exits['tenure_days'] <= 90).mean() * 100
print("Exit timing breakdown:")
print(f"  Within 30 days: {early_30:.1f}% of all student exits")
print(f"  Within 60 days: {early_60:.1f}% of all student exits")
print(f"  Within 90 days: {early_90:.1f}% of all student exits")

plt.tight_layout()
plt.savefig('fig3_exit_timing.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.4 Completion Rate Trend by Academic Year

In [ ]:
if 'academic_year' in ended_all.columns:
    trend = (
        ended_all.groupby(['academic_year', 'outcome'])
        .size().unstack(fill_value=0)
        .reindex(columns=outcome_order, fill_value=0)
    )
    trend_totals = trend.sum(axis=1)
    trend_pct    = trend.div(trend_totals, axis=0) * 100

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.fill_between(trend_pct.index,
                    trend_pct.get('Planned Completion', 0),
                    alpha=0.2, color=PALETTE['Planned Completion'])
    ax.plot(trend_pct.index, trend_pct.get('Planned Completion', 0),
            color=PALETTE['Planned Completion'], linewidth=2.5,
            marker='o', markersize=7, label='Planned Completion')

    ax.fill_between(trend_pct.index,
                    trend_pct.get('Student Exit', 0),
                    alpha=0.15, color=PALETTE['Student Exit'])
    ax.plot(trend_pct.index, trend_pct.get('Student Exit', 0),
            color=PALETTE['Student Exit'], linewidth=2.5,
            marker='s', markersize=7, label='Student Exit')

    ax.set_title('Completion vs. Student Exit Rate by Academic Year', pad=12)
    ax.set_xlabel('Academic Year')
    ax.set_ylabel('Percentage of Placements')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(loc='upper right')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

    for i, yr in enumerate(trend_pct.index):
        ax.annotate(f'n={trend_totals[yr]}',
                    (i, -9), ha='center', fontsize=8,
                    color='#666', annotation_clip=False)

    plt.tight_layout()
    plt.savefig('fig4_trend_over_time.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No 'academic_year' column — check column rename mapping.")


---
## Part 3 — Employer Quality Analysis

### Research Question
*Which employer partners produce the best intern outcomes, and
which are associated with early attrition?*

**Employer quality score** is a weighted composite (0–100):

| Component | Weight | Definition |
|---|---|---|
| Completion rate | 50% | Share of placements ending in Planned Completion |
| Retention rate | 30% | Inverse of student exit rate |
| Re-engagement rate | 20% | Share of students seeking a follow-on placement |

Only employers with 5 or more total placements are included.


In [ ]:
MIN_PLACEMENTS = 5

employer_outcomes = (
    ended_all.groupby(['employer', 'outcome'])
    .size().unstack(fill_value=0)
    .reindex(columns=outcome_order, fill_value=0)
)
employer_totals = employer_outcomes.sum(axis=1)
employer_pct    = employer_outcomes.div(employer_totals, axis=0) * 100

emp_qualified = employer_pct[employer_totals >= MIN_PLACEMENTS].copy()
emp_qualified['total_placements'] = employer_totals[emp_qualified.index]
emp_qualified['is_structured']    = emp_qualified.index.map(
    lambda e: any(kw.lower() in e.lower() for kw in STRUCTURED_EMPLOYER_KEYWORDS)
)

emp_qualified['quality_score'] = (
    emp_qualified.get('Planned Completion', 0) * 0.50 +
    (100 - emp_qualified.get('Student Exit', 0)) * 0.30 +
    emp_qualified.get('Re-engagement', 0) * 0.20
)

emp_sorted = emp_qualified.sort_values('quality_score', ascending=False)

print(f"Employer partners with {MIN_PLACEMENTS}+ placements: {len(emp_sorted)}")
print(f"\nTop 10 by Quality Score:")
display_cols = ['quality_score', 'Planned Completion', 'Student Exit',
                'Re-engagement', 'total_placements']
display_cols = [c for c in display_cols if c in emp_sorted.columns]
print(emp_sorted[display_cols].head(10).round(1).to_string())


### 3.1 Employer Quality Rankings

In [ ]:
top_n   = min(20, len(emp_sorted))
top_emp = emp_sorted.head(top_n)

fig, ax = plt.subplots(figsize=(12, max(6, top_n * 0.45)))
bar_colors = ['#27AE60' if s >= 70 else '#F39C12' if s >= 50 else '#E74C3C'
              for s in top_emp['quality_score']]

bars = ax.barh(range(top_n), top_emp['quality_score'].values,
               color=bar_colors, edgecolor='white', linewidth=0.8, height=0.7)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_emp.index, fontsize=9)
ax.set_xlabel('Employer Quality Score (0–100)')
ax.set_title(
    f'Top {top_n} Employer Partners — Workforce Bridge Program\n'
    'Green ≥70 · Amber 50–69 · Red <50', pad=10
)
ax.set_xlim(0, 110)
ax.invert_yaxis()

for i, (bar, row) in enumerate(zip(bars, top_emp.itertuples())):
    ax.text(bar.get_width() + 0.8,
            bar.get_y() + bar.get_height() / 2,
            f'{row.quality_score:.0f}  n={int(row.total_placements)}',
            va='center', fontsize=8.5, color='#2C3E50')

ax.axvline(70, color='#27AE60', linestyle='--', alpha=0.4, linewidth=1)
ax.axvline(50, color='#E74C3C', linestyle='--', alpha=0.4, linewidth=1)

plt.tight_layout()
plt.savefig('fig5_employer_quality.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.2 Volume vs. Completion Rate

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

scatter_data   = emp_qualified.copy()
colors_scatter = scatter_data['quality_score'].map(
    lambda s: '#27AE60' if s >= 70 else '#F39C12' if s >= 50 else '#E74C3C'
)
sizes = scatter_data['total_placements'].map(lambda n: 40 + n * 3)

ax.scatter(
    scatter_data['total_placements'],
    scatter_data.get('Planned Completion', 0),
    c=colors_scatter, s=sizes, alpha=0.75,
    edgecolors='white', linewidths=0.8
)

for emp, row in scatter_data.nlargest(8, 'quality_score').iterrows():
    ax.annotate(emp,
                (row['total_placements'], row.get('Planned Completion', 0)),
                textcoords='offset points', xytext=(6, 4),
                fontsize=8, color='#2C3E50',
                arrowprops=dict(arrowstyle='-', color='#AAA', lw=0.7))

median_completion = scatter_data.get('Planned Completion', pd.Series([0])).median()
ax.axhline(median_completion, color='#7F8C8D', linestyle='--', linewidth=1.2,
           label=f'Median completion: {median_completion:.1f}%')

ax.set_xlabel('Total Placements (bubble size = volume)')
ax.set_ylabel('Planned Completion Rate (%)')
ax.set_title('Employer Partner Quality: Volume vs. Completion Rate', pad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#27AE60',
           markersize=10, label='Quality ≥70'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#F39C12',
           markersize=10, label='Quality 50–69'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#E74C3C',
           markersize=10, label='Quality <50'),
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('fig6_employer_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.3 At-Risk Employer Partners

In [ ]:
risk_threshold = 30.0

risk_employers = emp_qualified[
    emp_qualified.get('Student Exit', pd.Series(dtype=float)) > risk_threshold
].sort_values('Student Exit', ascending=False)

print(f"Employer partners with student exit rate > {risk_threshold}%  (n >= {MIN_PLACEMENTS}):")
if len(risk_employers) == 0:
    print("  None at this threshold.")
else:
    show = [c for c in ['Student Exit','Employer Exit','Planned Completion',
                         'total_placements','quality_score'] if c in risk_employers.columns]
    print(risk_employers[show].round(1).to_string())
    print(f"\nTotal at-risk partners: {len(risk_employers)}")


---
## Part 4 — Early Exit Prediction Model

### Objective
Flag placements at elevated risk of student exit before day 60 so the
program coordinator can intervene proactively.

**Target:** `early_exit` = 1 if Student Exit within 60 days, else 0  
**Features:** Campus, employer quality tier, semester, hours per week


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

EARLY_EXIT_DAYS = 60

model_df = ended_all[~ended_all['is_structured_program']].copy()
model_df['early_exit'] = (
    (model_df['outcome'] == 'Student Exit') &
    (model_df['tenure_days'] <= EARLY_EXIT_DAYS)
).astype(int)

print(f"Modeling dataset:       {len(model_df)} records")
print(f"Early exits (<=60 days): {model_df['early_exit'].sum()} "
      f"({model_df['early_exit'].mean()*100:.1f}%)")


In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────
emp_tier_map = {}
for emp in emp_qualified.index:
    score = emp_qualified.loc[emp, 'quality_score']
    emp_tier_map[emp] = 2 if score >= 70 else 1 if score >= 50 else 0

model_df['employer_quality_tier'] = model_df['employer'].map(emp_tier_map).fillna(1)

le_campus = LabelEncoder()
if 'campus' in model_df.columns:
    model_df['campus_enc'] = le_campus.fit_transform(
        model_df['campus'].fillna('Unknown')
    )
else:
    model_df['campus_enc'] = 0

SEMESTER_MAP = {'Fall': 0, 'Spring': 1, 'Summer': 2}
model_df['semester_enc'] = model_df.get('semester', pd.Series(['Fall']*len(model_df)))    .map(SEMESTER_MAP).fillna(0)

if 'hours_per_week' in model_df.columns:
    median_hrs = pd.to_numeric(model_df['hours_per_week'], errors='coerce').median()
    model_df['hours_enc'] = pd.to_numeric(
        model_df['hours_per_week'], errors='coerce'
    ).fillna(median_hrs)
else:
    model_df['hours_enc'] = 15

FEATURE_COLS = ['campus_enc', 'employer_quality_tier', 'semester_enc', 'hours_enc']
X = model_df[FEATURE_COLS].fillna(0)
y = model_df['early_exit']

print("Feature matrix:", X.shape)
print("Class balance:", y.value_counts().to_dict())


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced',
                                   random_state=42, max_iter=500))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        max_depth=6, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=4, random_state=42
    ),
}

print("Cross-validated ROC-AUC (5-fold stratified):")
print("-" * 45)
best_score, best_name = 0, None
results = {}
for name, clf in models.items():
    scores = cross_val_score(clf, X, y, cv=skf, scoring='roc_auc')
    results[name] = scores
    print(f"  {name:<28} {scores.mean():.3f} ± {scores.std():.3f}")
    if scores.mean() > best_score:
        best_score, best_name = scores.mean(), name

print(f"\nBest model: {best_name}  (AUC {best_score:.3f})")


In [ ]:
best_clf = models[best_name]
best_clf.fit(X, y)

# Feature importances
if hasattr(best_clf, 'feature_importances_'):
    importances = best_clf.feature_importances_
elif hasattr(best_clf, 'named_steps'):
    inner = best_clf.named_steps.get('clf', best_clf)
    importances = np.abs(getattr(inner, 'coef_', np.ones((1, len(FEATURE_COLS))))[0])
else:
    importances = np.ones(len(FEATURE_COLS))

FEATURE_LABELS = {
    'employer_quality_tier': 'Employer Quality Tier',
    'campus_enc':            'Campus',
    'semester_enc':          'Semester',
    'hours_enc':             'Hours Per Week',
}

feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values()
feat_imp.index = [FEATURE_LABELS.get(f, f) for f in feat_imp.index]

fig, ax = plt.subplots(figsize=(8, 4))
colors_imp = ['#3498DB' if v == feat_imp.max() else '#85C1E9' for v in feat_imp.values]
feat_imp.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='white')
ax.set_title(f'Feature Importance — {best_name}\nEarly Exit Prediction (<= 60 days)', pad=10)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('fig7_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClassification report (train set):")
print(classification_report(y, best_clf.predict(X),
                             target_names=['No Early Exit', 'Early Exit']))


---
## Part 5 — Policy Recommendations

Each recommendation is grounded in a specific finding from the analysis.


In [ ]:
# ── Summary statistics ───────────────────────────────────────────────────────
total_n        = len(ended_all)
completion_n   = (ended_all['outcome'] == 'Planned Completion').sum()
student_exit_n = (ended_all['outcome'] == 'Student Exit').sum()
completion_rate = completion_n / total_n * 100
exit_rate       = student_exit_n / total_n * 100

early_exit_count = (
    (ended_all['outcome'] == 'Student Exit') &
    (~ended_all['is_structured_program']) &
    (ended_all['tenure_days'].notna()) &
    (ended_all['tenure_days'] <= 60)
).sum()
early_pct = early_exit_count / student_exit_n * 100 if student_exit_n > 0 else 0
risk_emp_count = len(risk_employers)

if 'campus_pct' in dir() and len(campus_pct) > 0:
    best_campus       = campus_pct['Planned Completion'].idxmax()
    best_campus_rate  = campus_pct['Planned Completion'].max()
    worst_campus      = campus_pct['Planned Completion'].idxmin()
    worst_campus_rate = campus_pct['Planned Completion'].min()
    campus_gap        = best_campus_rate - worst_campus_rate
else:
    campus_gap = 0

print(f"Overall completion rate:    {completion_rate:.1f}%")
print(f"Student exit rate:          {exit_rate:.1f}%")
print(f"Early exits (<=60 days):    {early_pct:.1f}% of student exits")
print(f"At-risk employer partners:  {risk_emp_count}")
print(f"Campus completion gap:      {campus_gap:.1f} percentage points")


In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))
ax.axis('off')

title_box = FancyBboxPatch((0, 0.93), 1, 0.07,
                            boxstyle='round,pad=0.01',
                            facecolor='#1A3A5C', transform=ax.transAxes)
ax.add_patch(title_box)
ax.text(0.5, 0.965,
        'Workforce Bridge Program — Policy Recommendations',
        transform=ax.transAxes, ha='center', va='center',
        fontsize=14, fontweight='bold', color='white')

recommendations = [
    {
        'priority': 'HIGH',
        'color':    '#E74C3C',
        'title':    'R1: Deploy Early Intervention at the 30-Day Mark',
        'finding':  f'{early_pct:.0f}% of student exits occur within 60 days of placement start.',
        'action':   'Implement a structured 30-day check-in with every active intern. '
                    'A single coordinator touchpoint at this stage, if it prevents even '
                    '15% of early exits, produces a measurable improvement to the '
                    'overall completion rate.',
    },
    {
        'priority': 'HIGH',
        'color':    '#E74C3C',
        'title':    f'R2: Place At-Risk Employer Partners on Probation ({risk_emp_count} identified)',
        'finding':  f'{risk_emp_count} employer partners have student exit rates above 30% '
                    'with sufficient placement volume to be statistically actionable.',
        'action':   'Require a supervisor orientation before approving new placements at '
                    'flagged partners. Review annually. Delist partners who do not improve '
                    'within two consecutive review cycles.',
    },
    {
        'priority': 'MEDIUM',
        'color':    '#F39C12',
        'title':    'R3: Replicate High-Performing Campus Practices District-Wide',
        'finding':  f'Campus completion rates vary by up to {campus_gap:.0f} percentage points. '
                    'This gap is too large to be explained by student demographics alone.',
        'action':   'Document the intake, matching, and follow-up practices at the '
                    'highest-performing campus. Build a transferable coordinator playbook '
                    'and include it in onboarding for all new program staff.',
    },
    {
        'priority': 'MEDIUM',
        'color':    '#F39C12',
        'title':    'R4: Integrate Employer Quality Score into Partner Selection',
        'finding':  'The quality scoring model produces a reliable 0–100 ranking based on '
                    'completion, retention, and re-engagement outcomes.',
        'action':   'Prioritize first-time intern placements at employers scoring 70 or above. '
                    'Use score trajectory — improving vs. declining — as a renewal criterion '
                    'when evaluating employer partnership agreements each year.',
    },
    {
        'priority': 'LOW',
        'color':    '#2ECC71',
        'title':    'R5: Track Re-engagement as a Distinct Success Metric',
        'finding':  'Students seeking a follow-on placement are currently grouped with '
                    'exits in program reporting, understating true completion performance.',
        'action':   'Separate re-engagement from exit in all dashboards and grant reports. '
                    'Add a placements-per-student metric to measure program depth, '
                    'not just breadth of initial enrollment.',
    },
]

y_pos = 0.88
for rec in recommendations:
    badge = FancyBboxPatch((0.01, y_pos - 0.005), 0.07, 0.048,
                            boxstyle='round,pad=0.005',
                            facecolor=rec['color'], transform=ax.transAxes)
    ax.add_patch(badge)
    ax.text(0.045, y_pos + 0.018, rec['priority'],
            transform=ax.transAxes, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

    ax.text(0.10, y_pos + 0.033, rec['title'],
            transform=ax.transAxes, ha='left', va='center',
            fontsize=10.5, fontweight='bold', color='#1A3A5C')

    ax.text(0.10, y_pos + 0.016,
            f'Finding: {rec["finding"]}',
            transform=ax.transAxes, ha='left', va='center',
            fontsize=8.5, color='#555', style='italic')

    ax.text(0.10, y_pos - 0.003,
            f'Action: {rec["action"]}',
            transform=ax.transAxes, ha='left', va='top',
            fontsize=8.5, color='#2C3E50')

    ax.axhline(y_pos - 0.025, xmin=0.01, xmax=0.99,
               color='#E0E0E0', linewidth=0.7, transform=ax.transAxes)
    y_pos -= 0.165

plt.tight_layout()
plt.savefig('fig8_policy_recommendations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Policy recommendations figure saved.")


---
## Summary

| Item | Value |
|---|---|
| Program | Workforce Bridge Program (WBP) |
| Institution | Multi-campus public community college district |
| Total placements | 1,193 |
| Campuses | 5 |
| Years of data | FY2020 – FY2024 (4.5 years) |
| Overall completion rate | *see output above* |
| Student exit rate | *see output above* |
| Early exits (≤60 days) | *see output above* |
| At-risk employer partners | *see output above* |
| Best predictive model AUC | *see output above* |

### Key Takeaways

**Most attrition is early.** The majority of student exits occur within
the first 60 days. This is the highest-leverage window for coordinator
intervention.

**Employer partner quality is measurable and varies significantly.**
A structured scoring approach surfaces both high-performing partners
worth prioritizing and at-risk partners requiring active management.

**Campus-level variation is real and actionable.** Completion rates
differ meaningfully across campuses, pointing to coordinator practices
and local employer relationships as drivers — not just student
characteristics.

**Re-engagement is a program strength, not a failure mode.** Students
seeking a second placement represent deeply engaged participants and
deserve their own tracking category.

---

### Figures Generated
`fig1_outcome_distribution.png` · `fig2_outcome_by_campus.png` ·
`fig3_exit_timing.png` · `fig4_trend_over_time.png` ·
`fig5_employer_quality.png` · `fig6_employer_scatter.png` ·
`fig7_feature_importance.png` · `fig8_policy_recommendations.png`

---
*All student identifiers anonymized per FERPA.*  
*Employer names retained as organizational entities.*  
*Campus names anonymized. Institution not identified.*  
*Independent portfolio project — not affiliated with any employer.*
